# Group 151: AUTOSAR Document Intelligence Assistant

AIMLCZG546 - Software Engineering for Machine Learning, Assignment I.

This notebook summarizes the implemented Retrieval-Augmented Generation (RAG) system, verifies the local API, and demonstrates retrieval over ingested AUTOSAR specifications.


## Group Details and Contributions

| # | BITS ID | Name | Qualitative Contribution | Contribution % |
|---|---|---|---|---:|
| 1 | 2025aa05473 | Abhinav Mandloi | Project integration, FastAPI gateway, report packaging, final verification | 30 |
| 2 | 2025aa05686 | Pritish Joshi | Retrieval pipeline, vector search, RAG prompt and citations | 25 |
| 3 | 2025aa05553 | Satwinder Singh | Quality requirements, health monitoring, heartbeat, reliability checks | 23 |
| 4 | 2025aa05533 | Shray Vijay | UI, feedback loop, testing, screenshots, documentation support | 22 |
| **Total** |  |  |  | **100** |


## Assignment Objective Mapping

Objective 1 is addressed through the GR4ML requirements, Business View, Analytics Design View, Data Preparation View, and the top three quality requirements in `docs/`.

Objective 2 is addressed through the HLD architecture diagram, RAG architectural pattern, service-oriented API decomposition, heartbeat/circuit-breaker reliability tactic, and the working FastAPI implementation.


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
required_paths = [
    ROOT / "README.md",
    ROOT / "HLD.png",
    ROOT / "docs" / "GR4ML_Requirements.md",
    ROOT / "docs" / "Business_View.md",
    ROOT / "docs" / "Analytics_Design_View.md",
    ROOT / "docs" / "Data_Preparation_View.md",
    ROOT / "docs" / "Quality_Requirements.md",
    ROOT / "docs" / "Architecture_Patterns.md",
    ROOT / "static" / "index.html",
]
for path in required_paths:
    print(f"{path.relative_to(ROOT)}: {'OK' if path.exists() else 'MISSING'}")


## Ingested AUTOSAR Corpus

The prepared vector store indexes five AUTOSAR PDFs. The metadata below is produced by the application ingestion registry.


In [ ]:
metadata_path = ROOT / "data" / "metadata" / "documents.json"
metadata = json.loads(metadata_path.read_text())
print(f"Documents indexed: {len(metadata)}")
print(f"Total chunks: {sum(doc['chunk_count'] for doc in metadata.values())}")
for name, doc in metadata.items():
    print(f"- {name}: {doc['page_count']} pages, {doc['chunk_count']} chunks")


## API Smoke Test

Start the app before running this section:

```bash
python -m uvicorn app.main:app --reload --port 8000
```


In [ ]:
import urllib.request
import urllib.error

BASE_URL = "http://127.0.0.1:8000"

def get_json(path):
    with urllib.request.urlopen(BASE_URL + path, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))

health = get_json("/health")
print("Overall status:", health["status"])
for service, detail in health["services"].items():
    print(f"{service}: {detail['status']}")


In [ ]:
query_payload = json.dumps({
    "question": "What is AUTOSAR layered software architecture?",
    "top_k": 3
}).encode("utf-8")
request = urllib.request.Request(
    BASE_URL + "/query/search",
    data=query_payload,
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=60) as response:
    search_result = json.loads(response.read().decode("utf-8"))

print("Results:", search_result["total_results"])
print("Latency ms:", search_result["latency_ms"])
for item in search_result["results"]:
    print(f"- {item['document']} p.{item['page']} score={item['similarity_score']}")


## Final Notes

- The implementation uses local Ollama models only, so AUTOSAR documents do not leave the machine.
- `/query/search` demonstrates fast retrieval from ChromaDB. Full `/query` answer generation is local-LLM dependent and returns measured `latency_ms`.
- Final report artifacts are provided as `151.docx` and `151.pdf`.
